# mpH2MM / smFRET Analysis — PTU files (Alexa555 / Alexa647)

**Donor:** Alexa 555 (excitation ~520 nm, emission ~570 nm)  
**Acceptor:** Alexa 647 (excitation ~640 nm, emission ~690 nm)  

**Pipeline:**
1. Convert `.ptu` → Photon-HDF5 via `phconvert` (aligned with `convert_ptus.py`)
2. Load & burst search with parameters from `calculate_BVA.py`
3. Burst selection: naa ≥ 30, size ≥ 150, S ∈ [0.25, 0.85]
4. First-round H²MM — identify and remove donor-only bursts
5. Second-round H²MM on FRET-active bursts
6. Dwell-time extraction and survival plots
7. BVA (chunk_size = 7, matching `calculate_BVA.py`)

**Key parameter alignment with reference scripts:**
- `convert_ptus.py` → donor=1 (Alexa555), acceptor=0 (Alexa647), ALEX periods (0,1000)/(1000,2000), overflow filter on nanotimes≠0
- `calculate_BVA.py` → `bg.exp_hist_fit`, time_s=60, F_bg=1.7, m=15, `Ph_sel(Dex='DAem')`, naa≥30, size≥150, S∈[0.25,0.85], BVA n=7
- `search_all.py` → L=50, M=35, T=500e-6 for DD/DA streams; L=50, M=15, T=500e-6 for AA stream


## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import shutil
from pathlib import Path

import fretbursts as frb
import phconvert as phc
import burstH2MM as bhm
import H2MM_C as h2

print('NumPy      :', np.__version__)
print('FRETbursts :', frb.__version__)
print('phconvert  :', phc.__version__)
print('burstH2MM  :', bhm.__version__)
print('H2MM_C     :', h2.__version__)


## 2 · PTU → Photon-HDF5 conversion helper

Aligned with `convert_ptus.py`:
- `donor=1` (Alexa555 emission channel), `acceptor=0` (Alexa647 emission channel)
- ALEX period donor `(0, 1000)`, acceptor `(1000, 2000)` ticks
- Overflow photons filtered out (`nanotimes != 0`) before saving
- `dispcurve` and `imghdr` metadata stripped to avoid HDF5 save errors


In [ ]:
def ptu_to_hdf5(
    ptu_file: str,
    output_path: str,
    description: str = 'smFRET measurement — Alexa555 / Alexa647',
    author: str = 'Fatma',
    author_affiliation: str = 'KUL',
    sample_name: str = 'sample',
    buffer_name: str = 'Tris',
    dye_names: str = 'alexa 555, alexa 647',
):
    """
    Convert a PicoQuant .ptu file to Photon-HDF5 format.

    Aligned with convert_ptus.py:
    - donor=1  (Alexa555 emission → detector 1)
    - acceptor=0 (Alexa647 emission → detector 0)
    - ALEX periods: donor=(0,1000), acceptor=(1000,2000) ticks
    - Overflow photons (nanotimes == 0) removed before saving
    - excitation_wavelengths=(520e-9, 640e-9)
    - detection_wavelengths=(570e-9, 690e-9)
    """
    print(f'Reading {ptu_file} ...')
    d, meta = phc.loader.nsalex_pq(
        str(ptu_file),
        donor=1,
        acceptor=0,
        alex_period_donor=(0, 1000),
        alex_period_acceptor=(1000, 2000),
        excitation_wavelengths=(520e-9, 640e-9),
        detection_wavelengths=(570e-9, 690e-9),
    )

    nanotimes  = d['photon_data']['nanotimes']
    detectors  = d['photon_data']['detectors']
    timestamps = d['photon_data']['timestamps']

    # Remove overflow photons (nanotime == 0 are sync/IRF artifacts)
    not_overflow = nanotimes != 0
    print(f'Removing {(~not_overflow).sum()} overflow photons '
          f'({(~not_overflow).mean()*100:.2f}%)')
    d['photon_data']['nanotimes']  = nanotimes[not_overflow]
    d['photon_data']['detectors']  = detectors[not_overflow]
    d['photon_data']['timestamps'] = timestamps[not_overflow]

    # Diagnostic: detector counts
    unique_det, counts = np.unique(d['photon_data']['detectors'], return_counts=True)
    print('Detectors after filter:', dict(zip(unique_det.tolist(), counts.tolist())))

    # TCSPC decay plot — verify ALEX period windows
    fig, ax = plt.subplots(figsize=(8, 3))
    for det_id in unique_det:
        mask = d['photon_data']['detectors'] == det_id
        ax.hist(d['photon_data']['nanotimes'][mask], bins=256,
                histtype='step', label=f'det {det_id}')
    ax.set_yscale('log')
    ax.set_xlabel('nanotime (ticks)')
    ax.set_ylabel('counts')
    ax.legend()
    ax.set_title('TCSPC decays — verify donor=(0,1000) & acceptor=(1000,2000)')
    ax.axvspan(0, 1000, alpha=0.1, color='green', label='donor window')
    ax.axvspan(1000, 2000, alpha=0.1, color='red', label='acceptor window')
    plt.tight_layout()
    plt.show()

    d['description'] = description
    d['sample'] = dict(
        sample_name=sample_name,
        dye_names=dye_names,
        buffer_name=buffer_name,
        num_dyes=len(dye_names.split(',')),
    )
    d['identity'] = dict(author=author, author_affiliation=author_affiliation)

    # Strip metadata that causes HDF5 save errors (from convert_ptus.py)
    _ = meta.pop('dispcurve', None)
    _ = meta.pop('imghdr', None)
    d['user'] = {'picoquant': meta}

    phc.hdf5.save_photon_hdf5(d, overwrite=True)
    print(f'Saved Photon-HDF5 → {output_path}')


## 3 · Filter junk detectors

Keeps only det 0 (Alexa647) and det 1 (Alexa555).  
Removes spurious channels (e.g. det 68, det 127 — sync/IRF artifacts).


In [ ]:
def filter_detectors(input_h5: str, output_h5: str, keep: tuple = (0, 1)):
    """
    Copy HDF5 keeping only photons from detectors in `keep`.
    Removes junk channels (e.g. det 68, 127 sync/IRF artifacts).
    """
    shutil.copy(input_h5, output_h5)
    with h5py.File(output_h5, 'r+') as f:
        det = f['photon_data/detectors'][:]
        ts  = f['photon_data/timestamps'][:]
        mask = np.zeros(len(det), dtype=bool)
        for d in keep:
            mask |= (det == d)
        print(f'Keeping {mask.sum()} of {len(mask)} photons '
              f'({mask.sum()/len(mask)*100:.1f}%) — detectors {keep}')
        for key, arr in [('detectors', det), ('timestamps', ts)]:
            del f[f'photon_data/{key}']
            f.create_dataset(f'photon_data/{key}', data=arr[mask])
    print(f'Filtered HDF5 saved → {output_h5}')


## 4 · Burst search helper

Parameters aligned with **`calculate_BVA.py`**:
- Background: `bg.exp_hist_fit`, `time_s=60`, `F_bg=1.7`  
- Burst search: `m=15`, photon selection `Ph_sel(Dex='DAem')`  
- Burst selection: `naa ≥ 30` → `size ≥ 150` → `S ∈ [0.25, 0.85]`

> **Note:** `search_all.py` uses `dont_fret` with `L=50, M=35, T=500e-6` (DD/DA)  
> and `L=50, M=15, T=500e-6` (AA) — these are equivalent burst-search criteria  
> expressed in the FRETbursts framework as `m=15, F=6.0`.


In [ ]:
def load_and_burst_search(
    h5_file: str,
    D_ON: tuple = (0, 1000),
    A_ON: tuple = (1000, 2000),
    m: int = 15,
    F: float = 6.0,
    min_naa: int = 30,
    min_burst_size: int = 150,
    S1: float = 0.25,
    S2: float = 0.85,
):
    """
    Load Photon-HDF5, apply ALEX periods, run burst search and selection.

    Parameters aligned with calculate_BVA.py:
      - bg.exp_hist_fit, time_s=60, F_bg=1.7
      - m=15 (min photons for burst start), Ph_sel(Dex='DAem')
      - naa >= 30, size >= 150, S in [0.25, 0.85]

    Parameters
    ----------
    D_ON / A_ON    : ALEX excitation windows in timestamp ticks
    m              : min consecutive Dex photons for burst start (15)
    F              : burst threshold multiplier (6.0 x background)
    min_naa        : min Aex-Aem photons per burst (30, from calculate_BVA.py)
    min_burst_size : min total burst size in photons (150, from calculate_BVA.py)
    S1 / S2        : stoichiometry filter range (0.25–0.85, from calculate_BVA.py)

    Returns
    -------
    data     : raw FRETbursts Data object (all bursts)
    data_sel : burst-selected Data object after naa + size + S filters
    """
    data = frb.loader.photon_hdf5(h5_file)
    data.add(
        donor=1,                      # Alexa555 emission → detector 1
        acceptor=0,                   # Alexa647 emission → detector 0
        alex_period_donor=D_ON,
        alex_period_acceptor=A_ON,
    )
    frb.loader.alex_apply_period(data)

    # Background — aligned with calculate_BVA.py: exp_hist_fit, 60s, F_bg=1.7
    data.calc_bg(fun=frb.bg.exp_hist_fit, time_s=60, tail_min_us='auto', F_bg=1.7)

    # Burst search on Dex stream only (Ph_sel(Dex='DAem')) — calculate_BVA.py
    data.burst_search(m=m, F=F, ph_sel=frb.Ph_sel(Dex='DAem'), computefret=False)
    data.calc_fret(count_ph=True, corrections=False)

    # Selection cascade from calculate_BVA.py:
    #   1. naa >= 30
    data_sel = data.select_bursts(frb.select_bursts.naa, th1=min_naa, computefret=False)
    #   2. size >= 150
    data_sel = data_sel.select_bursts(frb.select_bursts.size, th1=min_burst_size, computefret=False)
    #   3. S in [0.25, 0.85]
    data_sel = data_sel.select_bursts(frb.select_bursts.S, S1=S1, S2=S2, computefret=False)

    print(f'Total bursts            : {data.mburst[0].num_bursts}')
    print(f'After naa >= {min_naa}        : {data.select_bursts(frb.select_bursts.naa, th1=min_naa, computefret=False).mburst[0].num_bursts}')
    print(f'After size >= {min_burst_size}      : {data.select_bursts(frb.select_bursts.naa, th1=min_naa, computefret=False).select_bursts(frb.select_bursts.size, th1=min_burst_size, computefret=False).mburst[0].num_bursts}')
    print(f'After S in [{S1},{S2}] : {data_sel.mburst[0].num_bursts}')

    return data, data_sel


## 5 · BVA helpers

Aligned with **`calculate_BVA.py`**: chunk size `n=7`, uses `Ph_sel(Dex='DAem')` and `Ph_sel(Dex='Aem')`.  
Shot-noise reference: `y = sqrt(E*(1-E) / 7)`.


In [ ]:
# BVA chunk size — aligned with calculate_BVA.py (n=7)
BVA_CHUNK_SIZE = 7

def BVA(data, chunk_size: int = BVA_CHUNK_SIZE):
    """
    Burst Variance Analysis — compute sub-burst FRET std-dev per burst.

    Aligned with calculate_BVA.py:
    - Uses Ph_sel(Dex='DAem') for donor excitation photons
    - Uses Ph_sel(Dex='Aem')  for FRET (acceptor emission during donor excitation)
    - chunk_size=7 (n=7 in calculate_BVA.py)
    """
    E_eff, std_E = [], []
    for ich, mburst in enumerate(data.mburst):
        stdE, E = [], []
        Aem = data.get_ph_mask(ich=ich, ph_sel=frb.Ph_sel(Dex='Aem'))
        Dex = data.get_ph_mask(ich=ich, ph_sel=frb.Ph_sel(Dex='DAem'))
        for istart, istop in zip(mburst.istart, mburst.istop):
            phots = Aem[istart:istop+1][Dex[istart:istop+1]]
            Esub  = [phots[nb:ne].sum()
                     for nb, ne in zip(range(0, phots.size, chunk_size),
                                       range(chunk_size, phots.size+1, chunk_size))]
            if Esub:
                stdE.append(np.std(Esub) / chunk_size)
                E.append(sum(Esub) / (len(Esub) * chunk_size))
            else:
                stdE.append(np.nan)
                E.append(np.nan)
        E_eff.append(np.array(E))
        std_E.append(np.array(stdE))
    return E_eff, std_E


def bin_bva(E, std, R: int = 10, B_thr: int = 50):
    """Bin BVA results into R bins for a smoother plot."""
    E_cat   = np.concatenate(E)
    std_cat = np.concatenate(std)
    # Remove NaN
    valid   = np.isfinite(E_cat) & np.isfinite(std_cat)
    E_cat, std_cat = E_cat[valid], std_cat[valid]
    bn      = np.linspace(0, 1, R + 1)
    std_avg, E_avg = np.full(R, -1.0), np.full(R, -1.0)
    for i, (bb, be) in enumerate(zip(bn[:-1], bn[1:])):
        mask = (bb <= E_cat) & (E_cat < be)
        if mask.sum() > B_thr:
            std_avg[i] = np.mean(std_cat[mask])
            E_avg[i]   = np.mean(E_cat[mask])
    return E_avg, std_avg


# Shot-noise reference curve (chunk_size=7, matching calculate_BVA.py)
x_T = np.arange(0, 1.01, 0.01)
y_T = np.sqrt((x_T * (1 - x_T)) / BVA_CHUNK_SIZE)


## 6 · User input — set your PTU file and metadata

In [ ]:
# ── USER INPUT ────────────────────────────────────────────────────────────
ptu_file    = r'C:/Users/fatma/Dropbox/2026/PhD experiments/SmFRET/2605/260517_sFRET_TFAE_3a_4a_mutants.sptw/260517_sFRET_TFAE_3a_4000x_apo2.ptu'
output_h5   = 'measurement_raw.h5'         # raw HDF5 (before detector filter)
filtered_h5 = 'measurement_filtered.h5'    # filtered HDF5 (det 0 & 1 only)

sample_name  = ''      # e.g. 'APO', 'HOLO', '3a_4a_mutant'
buffer_name  = 'Tris'
author       = 'Fatma'
# ─────────────────────────────────────────────────────────────────────────

ptu_to_hdf5(
    ptu_file=ptu_file,
    output_path=output_h5,
    description='smFRET measurement — Alexa555 / Alexa647',
    author=author,
    author_affiliation='KUL',
    sample_name=sample_name,
    buffer_name=buffer_name,
    dye_names='alexa 555, alexa 647',
)


## 7 · Filter junk detectors (keep det 0 & 1 only)

In [ ]:
filter_detectors(output_h5, filtered_h5, keep=(0, 1))

## 8 · Burst search + burst selection

Uses parameters from **`calculate_BVA.py`**:
- Background: `exp_hist_fit`, 60 s windows, F_bg=1.7
- Search: m=15 on `Ph_sel(Dex='DAem')`
- Select: naa ≥ 30 → size ≥ 150 → S ∈ [0.25, 0.85]


In [ ]:
smfret_raw, smfret_sel = load_and_burst_search(
    h5_file=filtered_h5,
    D_ON=(0, 1000),      # donor excitation window — ticks (convert_ptus.py)
    A_ON=(1000, 2000),   # acceptor excitation window — ticks (convert_ptus.py)
    m=15,                # min Dex photons for burst start (calculate_BVA.py)
    F=6.0,               # burst threshold x background
    min_naa=30,          # min Aex-Aem photons (calculate_BVA.py: th1=30)
    min_burst_size=150,  # min burst size (calculate_BVA.py: th1=150)
    S1=0.25,             # stoichiometry lower bound (calculate_BVA.py)
    S2=0.85,             # stoichiometry upper bound (calculate_BVA.py)
)

frb.alex_jointplot(smfret_sel)
plt.title('After burst search + naa + size + S filter')
plt.show()


## 9 · First-round H²MM — identify donor-only bursts

Fits 1–6 state models. Check **ICL** and **BICp** plots:  
- Set `round1_idx` to the first model where BICp drops below 0.005  
- 0-indexed: `models[0]`=1-state, `models[1]`=2-state, etc.


In [ ]:
print('Fitting first-round H2MM models (this may take a few minutes)...')
smfret_mp = bhm.BurstData(smfret_sel)
smfret_mp.models.calc_models(max_state=6, max_iter=3600)
print('Done.')

bhm.ICL_plot(smfret_mp.models)
plt.title('ICL — first round')
plt.show()

bhm.BICp_plot(smfret_mp.models)
plt.axhline(0.005, color='red', linestyle='--', label='threshold 0.005')
plt.legend()
plt.title('BICp — first round (pick first model below red line)')
plt.show()


In [ ]:
# ── SET based on ICL/BICp plots above ─────────────────────────────────────
# 0-indexed: models[0]=1-state, models[1]=2-state, models[2]=3-state ...
round1_idx = 3   # ← adjust if needed
# ─────────────────────────────────────────────────────────────────────────

model_r1 = smfret_mp.models[round1_idx]
print(f'Model states : {model_r1.nstate}')
print(f'E per state  : {model_r1.E}')
print(f'S per state  : {model_r1.S}')

bhm.dwell_ES_scatter(model_r1, alpha=0.2, s=2)
bhm.trans_arrow_ES(model_r1)
plt.title(f'First-round H2MM — {model_r1.nstate} states')
plt.show()

# Identify donor-only state (highest S value → low E, high S)
donor_only_state = int(np.argmax(model_r1.S))
print(f'\nDonor-only state index: {donor_only_state}  '
      f'E={model_r1.E[donor_only_state]:.3f}  '
      f'S={model_r1.S[donor_only_state]:.3f}')

# Keep bursts that have at least one photon NOT in the donor-only state
fret_mask = np.array([
    not np.all(path == donor_only_state)
    for path in model_r1.path
])
print(f'Bursts before H2MM filter : {len(fret_mask)}')
print(f'Bursts after  H2MM filter : {fret_mask.sum()}')

smfret_fret = smfret_sel.select_bursts_mask_apply(np.where(fret_mask))

frb.alex_jointplot(smfret_fret)
plt.title('FRET-active bursts after first-round H2MM donor-only removal')
plt.show()


## 10 · Second-round H²MM — final model on FRET-active bursts

Set `final_idx` to the model suggested by the second-round BICp plot.


In [ ]:
print('Fitting second-round H2MM models on FRET-active bursts...')
smfret_fret_mp = bhm.BurstData(smfret_fret)
smfret_fret_mp.models.calc_models(max_state=6, max_iter=3600)
print('Done.')

bhm.ICL_plot(smfret_fret_mp.models)
plt.title('ICL — second round')
plt.show()

bhm.BICp_plot(smfret_fret_mp.models)
plt.axhline(0.005, color='red', linestyle='--', label='threshold 0.005')
plt.legend()
plt.title('BICp — second round (pick first model below red line)')
plt.show()


In [ ]:
# ── SET based on second-round BICp plot ───────────────────────────────────
final_idx = 4   # ← adjust (0-indexed)
# ─────────────────────────────────────────────────────────────────────────

model_final = smfret_fret_mp.models[final_idx]
print(f'Final model states : {model_final.nstate}')
print(f'E per state        : {model_final.E}')
print(f'S per state        : {model_final.S}')
print(f'Trans matrix       :\n{model_final.trans}')

fig, ax = plt.subplots(figsize=(5, 4))
bhm.dwell_ES_scatter(model_final, alpha=0.6, s=1)
bhm.trans_arrow_ES(model_final, min_rate=0)
bhm.scatter_ES(model_final, s=100, c='r')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel(r'$E$'); ax.set_ylabel(r'$S$')
plt.title(f'Final H2MM — {model_final.nstate} states')
plt.tight_layout()
plt.show()


## 11 · Dwell-time extraction and survival plots

In [ ]:
n_states = model_final.nstate
print(f'Total dwells        : {len(model_final.dwell_dur)}')
print(f'Dwell states present: {np.unique(model_final.dwell_state)}')

dwell_per_state = [
    model_final.dwell_dur[model_final.dwell_state == s]
    for s in range(n_states)
]

# Convert photons → ms using mean in-burst photon rate
bursts       = smfret_fret.mburst[0]
mean_ph_rate = bursts.counts.sum() / (bursts.width.sum() * smfret_fret.clk_p)
print(f'Mean in-burst rate  : {mean_ph_rate:.0f} ph/s')
print()

for s in range(n_states):
    dt_ms = dwell_per_state[s] / mean_ph_rate * 1000
    print(f'State {s} (E≈{model_final.E[s]:.2f}): '
          f'{len(dwell_per_state[s])} dwells  '
          f'mean={dt_ms.mean():.3f} ms  '
          f'max={dt_ms.max():.3f} ms')

# Survival function plots
fig, axes = plt.subplots(1, n_states, figsize=(4*n_states, 4), sharey=True)
if n_states == 1:
    axes = [axes]

for s, ax in enumerate(axes):
    dt   = dwell_per_state[s]
    dt   = dt[dt > 0]
    t_ms = np.sort(dt / mean_ph_rate * 1000)
    surv = 1 - np.arange(1, len(t_ms)+1) / len(t_ms)
    ax.semilogy(t_ms, surv, '.', markersize=3)
    ax.set_xlabel('Dwell time (ms)')
    ax.set_ylabel('Survival probability')
    ax.set_title(f'State {s}  E≈{model_final.E[s]:.2f}  (n={len(dt)})')

plt.suptitle('Dwell-time survival functions per H²MM state')
plt.tight_layout()
plt.show()


## 12 · BVA — Burst Variance Analysis

Aligned with **`calculate_BVA.py`**: chunk_size = 7.  
Points above the dashed shot-noise line confirm genuine conformational dynamics.


In [ ]:
E_bva, std_bva = BVA(smfret_fret, chunk_size=BVA_CHUNK_SIZE)
E_avg, std_avg = bin_bva(E_bva, std_bva, R=10, B_thr=50)

# Concatenate and remove NaN
E_cat   = np.concatenate(E_bva)
std_cat = np.concatenate(std_bva)
valid_pts = np.isfinite(E_cat) & np.isfinite(std_cat)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(E_cat[valid_pts], std_cat[valid_pts],
           s=2, alpha=0.3, color='grey', label='per burst')

valid_bins = E_avg > 0
ax.scatter(E_avg[valid_bins], std_avg[valid_bins],
           s=40, color='blue', zorder=5, label='binned avg')

# Shot-noise line (chunk_size=7, as in calculate_BVA.py)
ax.plot(x_T, y_T, 'k--', lw=2, label=f'shot noise (n={BVA_CHUNK_SIZE})')

ax.set_xlim(0, 1)
ax.set_ylim(0, np.sqrt(0.5**2 / BVA_CHUNK_SIZE) * 2)
ax.set_xlabel(r'$\langle E \rangle$')
ax.set_ylabel(r'$\sigma_E$')
ax.set_title(f'BVA (n={BVA_CHUNK_SIZE}) — points above dashed line = dynamics')
ax.text(0.05, 0.95, f'n = {valid_pts.sum()} bursts',
        va='top', transform=ax.transAxes)
ax.legend()
plt.tight_layout()
plt.show()
